# §1.6.1 — 세 오차 항을 각각 수치로 분리하기

> 딥러닝 교재 · 1부 1장 6절 1항 (🐍)
> 선행: §1.2.1(베이즈 위험) · §1.4.1(가설 공간과 분해) · §1.4.2(귀납 편향의 세 출처)

## 이 노트북이 답하는 질문

1. **세 항을 실제로 따로 잴 수 있는가?** 근사·추정·최적화 오차를 각각 수치로 분리한다.
2. **각 항은 무엇으로 줄어드는가?** 용량 · 자료 · 계산 세 축을 하나씩 훑는다.
3. **최적화 오차는 항상 양수인가?** 정확한 ERM 해보다 **덜 수렴한 해가 더 나은** 경우를 찾는다.
4. **세 항을 따로 조절할 수 있는가?** 자료를 늘리면 최적 용량이 어떻게 움직이는가.

**예상 실행 시간** CPU 단일 코어 약 40초 (`FAST = True`이면 약 15초).

§1.4.1은 분해식을 유도했지만 각 항이 실제로 얼마인지는 묻지 않았다. 이 노트북이 그 값을 잰다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial import legendre as npleg

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260802
SIGMA    = 0.3     # 잡음 표준편차. R* = SIGMA^2 이다
KMAX     = 400     # 참 함수 계수를 몇 차까지 계산할지 (g=|x|는 무한히 이어진다)
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi=[0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_1_6_1_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. 문제 설계 — 왜 이 문제인가

세 항을 **각각** 재려면 네 개의 함수를 전부 알아야 한다. 그중 둘($f_H$, $f_S$)은 보통 계산할 수 없다.
그래서 계산할 수 있는 문제를 일부러 만든다.

**자료 생성 분포.**

$$x \sim \mathrm{Uniform}(-1, 1), \qquad y = g(x) + \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

**손실은 제곱오차.** §1.2.2에 의해 $f^{*}(x) = g(x)$ 이고 $R^{*} = \sigma^2$ 이다.

**가설 공간은 차수 $k$ 다항식.** 다만 단항식 $1, x, x^2, \dots$ 대신 **정규직교 기저**를 쓴다.
균등분포 $[-1,1]$ 에 대해 르장드르 다항식 $P_j$ 를 정규화한

$$\varphi_j(x) \triangleq \sqrt{2j+1}\, P_j(x), \qquad \mathbb{E}[\varphi_i \varphi_j] = \delta_{ij}$$

가 그것이다. 기저를 바꾸는 것은 $H$ 를 바꾸지 않는다 — 차수 $k$ 다항식 전체로 같다.
(§1.4.3에서 기저가 **최적화의 선택**을 바꾸는 것은 보았다. 여기서는 $H$ 자체가 같다는 점이 요점이다.)

### 이 기저가 주는 것

$f_w = \sum_{j \le k} w_j \varphi_j$ 이고 $c_j \triangleq \mathbb{E}[g\,\varphi_j]$ 라 하면, 직교성에서 곧바로

$$R(f_w) - R^{*} = \mathbb{E}\big[(f_w(x) - g(x))^2\big] = \underbrace{\sum_{j \le k} (w_j - c_j)^2}_{\text{계수 오차}} + \underbrace{\sum_{j > k} c_j^2}_{\text{잘린 꼬리}}$$

**초과 위험이 계수 벡터의 거리로 떨어진다.** 적분도 몬테카를로도 필요 없다. 이것이 이 문제를 고른 이유다.

In [ ]:
def phi(x, k):
    # 정규직교 르장드르 설계행렬 (len(x), k+1)
    x = np.asarray(x, float)
    out = np.empty((len(x), k+1))
    for j in range(k+1):
        c = np.zeros(j+1); c[j] = 1.0
        out[:, j] = npleg.legval(x, c) * np.sqrt(2*j+1)
    return out

GX, GW = npleg.leggauss(1200)          # 가우스-르장드르 구적 (기댓값 = sum w/2 * f)
def expect(vals):
    return float(np.sum((GW/2.0) * vals))

# 직교성 확인
_P = phi(GX, 10)
_G = (_P * (GW/2)[:, None]).T @ _P
assert np.abs(_G - np.eye(11)).max() < 1e-10
print(f"정규직교성 확인: 최대 오차 {np.abs(_G-np.eye(11)).max():.2e}")

g = lambda x: np.abs(np.asarray(x, float))          # 참 함수
C = (phi(GX, KMAX) * (GW/2)[:, None]).T @ g(GX)     # c_j
print(f"참 함수 계수 c_j (0..7): {np.round(C[:8], 4)}")
print("-> 홀수 차수가 0인 것은 g가 우함수이기 때문")

# g=|x|의 전개는 무한히 이어지므로 KMAX에서 자른다. 그 잔여를 확인해 둔다.
_resid = expect(g(GX)**2) - float(np.sum(C**2))
print(f"\nKMAX={KMAX}에서 잘라낸 잔여 꼬리: {_resid:.2e}")
print("-> 이하의 '폐형식'은 이 크기만큼의 절단 오차를 갖는다")

---
## 2. 네 함수와 세 항

| 기호 | 무엇인가 | 이 문제에서 |
|---|---|---|
| $f^{*}$ | 베이즈 최적 | $g$ 자체. $R^{*} = \sigma^2$ |
| $f_H$ | $H$ 안의 최선 — $\arg\min_{f \in H} R(f)$ | 계수를 $c_j$ 로 자른 것 |
| $f_S$ | 정확한 ERM 해 — $\arg\min_{f \in H} \hat{R}_n(f)$ | 최소제곱 해 |
| $\hat{f}_T$ | 최적화기가 $T$ 걸음 뒤 내놓은 것 | 경사하강 |

세 항은 이 넷의 차이다.

$$R(\hat{f}_T) - R^{*} = \underbrace{\big[R(f_H) - R^{*}\big]}_{\text{근사}} + \underbrace{\big[R(f_S) - R(f_H)\big]}_{\text{추정}} + \underbrace{\big[R(\hat{f}_T) - R(f_S)\big]}_{\text{최적화}}$$

앞의 두 항은 항상 비음이다. **셋째는 부호가 정해져 있지 않다** — 5절에서 실제로 음수가 되는 것을 본다.

In [ ]:
def approx_err(k):
    # 근사 오차 = 잘린 꼬리 = R(f_H) - R*
    return float(np.sum(C[k+1:]**2))

def excess(w, k):
    # R(f_w) - R* = sum_{j<=k}(w_j-c_j)^2 + tail
    return float(np.sum((np.asarray(w) - C[:k+1])**2) + approx_err(k))

def sample(n, rng):
    x = rng.uniform(-1, 1, n)
    return x, g(x) + rng.normal(0, SIGMA, n)

def erm(x, y, k):
    return np.linalg.pinv(phi(x, k)) @ y          # 정확한 최소제곱 해 f_S

def gd_path(x, y, k, T):
    A = phi(x, k); n = len(x)
    lr = 1.0/np.linalg.eigvalsh((2.0/n)*(A.T @ A)).max()   # 안정 학습률
    w = np.zeros(k+1); out = np.empty(T)
    for t in range(T):
        w = w - lr*(2.0/n)*(A.T @ (A @ w - y))
        out[t] = excess(w, k)
    return out, w, lr

# ── 공식 검증: 계수 공식 vs 직접 수치적분 ──
_k = 6
_x, _y = sample(200, np.random.default_rng(SEED))
_w = erm(_x, _y, _k)
_direct = expect((phi(GX, _k) @ _w - g(GX))**2)
print(f"계수 공식      : {excess(_w,_k):.10f}")
print(f"직접 수치적분  : {_direct:.10f}")
assert abs(excess(_w,_k) - _direct) < 1e-7
print("일치 확인 -> 이후 모든 값은 폐형식으로 계산한다\n")

# ── 세 항이 실제로 합쳐지는지 ──
k, n, T = 12, 60, 3000
x, y = sample(n, np.random.default_rng(3))
wS = erm(x, y, k)
path, wT, lr = gd_path(x, y, k, T)
a = approx_err(k); e = excess(wS, k) - a; o = excess(wT, k) - excess(wS, k)
print(f"k={k}, n={n}, T={T}, 학습률={lr:.4f}")
print(f"  근사   R(f_H)-R*      = {a:.6f}")
print(f"  추정   R(f_S)-R(f_H)  = {e:.6f}")
print(f"  최적화 R(f_T)-R(f_S)  = {o:+.6f}")
print(f"  합                     = {a+e+o:.6f}")
print(f"  직접   R(f_T)-R*      = {excess(wT,k):.6f}")
assert abs((a+e+o) - excess(wT,k)) < 1e-12
print("\n세 항의 합이 총 초과 위험과 일치한다.")

In [ ]:
xg = np.linspace(-1, 1, 400)
fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.plot(xg, g(xg), color=CB[0], lw=2.0, label=lab('$f^{*} = g$ (베이즈 최적)', '$f^{*} = g$ (Bayes)'))
ax.plot(xg, phi(xg, k) @ C[:k+1], color=CB[3], lw=1.5, ls='--',
        label=lab('$f_H$ ($H$ 안의 최선)', '$f_H$ (best in $H$)'))
ax.plot(xg, phi(xg, k) @ wS, color=CB[4], lw=1.5,
        label=lab('$f_S$ (정확한 ERM)', '$f_S$ (exact ERM)'))
ax.plot(xg, phi(xg, k) @ wT, color=CB[5], lw=1.5, ls=':',
        label=lab(f'$\\hat{{f}}_T$ (GD, $T$={T})', f'$\\hat{{f}}_T$ (GD, $T$={T})'))
ax.plot(x, y, 'o', ms=3.5, color='0.6', label=lab('훈련 자료', 'training data'))
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(lab(f'네 함수 ($k$={k}, $n$={n})', f'the four functions ($k$={k}, $n$={n})'), fontsize=10)
ax.legend(fontsize=7.5); show('four_functions')

---
## 3. 용량 축 — $k$ 를 훑는다

$n$ 을 고정하고 가설 공간의 크기만 바꾼다. 예상은 명확하다.

- **근사 오차**는 $k$ 가 커지면 단조 감소한다 ($H_k \subseteq H_{k+1}$ 이므로 자명)
- **추정 오차**는 $k$ 가 커지면 증가한다 (추정할 계수가 늘어나므로)

잘 명세된 최소제곱에서 추정 오차의 이론값은 $\dfrac{(k+1)\sigma^2}{n}$ 이다. 실측과 대조한다.

In [ ]:
n3 = 200
KS = [0,1,2,3,4,6,8,10,12,16,20,24,28]
T3 = 20 if FAST else 60

approxs, ests = [], []
for kk in KS:
    ee = [excess(erm(*sample(n3, np.random.default_rng(100+t)), kk), kk) - approx_err(kk)
          for t in range(T3)]
    approxs.append(approx_err(kk)); ests.append(float(np.mean(ee)))
approxs, ests = np.array(approxs), np.array(ests)
totals = approxs + ests
theory = np.array([(kk+1)*SIGMA**2/n3 for kk in KS])

print(f"n = {n3}")
print("  k     근사      추정(실측)  추정(이론)     합")
for i, kk in enumerate(KS):
    print(f"{kk:>3}  {approxs[i]:.6f}  {ests[i]:.6f}   {theory[i]:.6f}  {totals[i]:.6f}")
kbest = KS[int(np.argmin(totals))]
print(f"\n총 초과 위험이 최소인 용량: k = {kbest}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.semilogy(KS, approxs, 'o-', color=CB[3], label=lab('근사 오차', 'approximation'))
ax.semilogy(KS, ests,    's-', color=CB[4], label=lab('추정 오차', 'estimation'))
ax.semilogy(KS, totals,  '^-', color=CB[0], lw=2, label=lab('합', 'total'))
ax.semilogy(KS, theory,  '--', color=CB[5], lw=1.2,
            label=lab(r'이론 $(k{+}1)\sigma^2/n$', r'theory $(k{+}1)\sigma^2/n$'))
ax.axvline(kbest, color=CB[1], lw=1.2, ls=':',
           label=lab(f'최소점 $k$ = {kbest}', f'minimum at $k$ = {kbest}'))
ax.set_xlabel(lab('가설 공간의 용량 $k$ (다항식 차수)', 'capacity $k$ (polynomial degree)'))
ax.set_ylabel(lab('초과 위험', 'excess risk'))
ax.set_title(lab(f'용량을 키우면 근사는 줄고 추정은 는다 ($n$ = {n3})',
                 f'capacity trades approximation against estimation ($n$ = {n3})'), fontsize=10)
ax.legend(fontsize=8); show('capacity_axis')

> **읽는 법.** 두 선이 반대 방향으로 움직이고 합이 U자를 그린다. 이것이 §1.4.1 분해의 앞 두 항이다.
>
> 이론선이 $k \lesssim 12$ 까지만 맞는 것도 의미가 있다. 그 너머에서는 표본 $n$ 개로 추정한 그람 행렬이
> 나빠져 실측이 이론을 크게 웃돈다. **점근 이론이 유한 표본에서 언제 깨지는지**를 보는 자리다.

---
## 4. 자료 축 — $n$ 을 훑는다

용량을 고정하고 표본 수만 바꾼다. **근사 오차는 $n$ 과 무관해야 한다** — $H$ 가 바뀌지 않았으므로.
움직이는 것은 추정 오차뿐이고, $\sigma^2 (k{+}1)/n$ 을 따라야 한다.

In [ ]:
k4 = 6
NS = [40, 80, 160, 320, 640, 1280] + ([] if FAST else [2560])
T4 = 20 if FAST else 60
est_n = []
for nn in NS:
    ee = [excess(erm(*sample(nn, np.random.default_rng(200+t)), k4), k4) - approx_err(k4)
          for t in range(T4)]
    est_n.append(float(np.mean(ee)))
est_n = np.array(est_n)
th_n  = np.array([(k4+1)*SIGMA**2/nn for nn in NS])
slope_all, _ = np.polyfit(np.log(NS), np.log(est_n), 1)
_ok = [i for i, nn in enumerate(NS) if nn >= 160]     # 조건수가 안정된 구간만
slope, _ = np.polyfit(np.log(np.array(NS)[_ok]), np.log(est_n[_ok]), 1)

print(f"k = {k4} 고정, 근사 오차 = {approx_err(k4):.6f} (n에 무관)")
print("    n     추정(실측)  추정(이론)   비")
for i, nn in enumerate(NS):
    print(f"{nn:>6}   {est_n[i]:.6f}   {th_n[i]:.6f}   {est_n[i]/th_n[i]:.3f}")
print(f"\n적합 기울기 (n>=160) = {slope:+.3f}   (이론 -1)")
print(f"적합 기울기 (전체)   = {slope_all:+.3f}   <- 작은 n에서 그람 행렬이 나빠 실측이 이론을 웃돈다")

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.8))
ax.loglog(NS, est_n, 'o-', color=CB[4], label=lab('추정 오차 (실측)', 'estimation (observed)'))
ax.loglog(NS, th_n, '--', color=CB[5], label=lab(r'이론 $(k{+}1)\sigma^2/n$', r'theory $(k{+}1)\sigma^2/n$'))
ax.axhline(approx_err(k4), color=CB[3], lw=1.5,
           label=lab(f'근사 오차 = {approx_err(k4):.5f} (고정)', f'approximation (constant)'))
ax.set_xlabel(lab('표본 수 $n$ (개)', 'sample size $n$'))
ax.set_ylabel(lab('초과 위험', 'excess risk'))
ax.set_title(lab(f'자료는 추정 오차만 줄인다 ($k$ = {k4})',
                 f'data reduces only the estimation term ($k$ = {k4})'), fontsize=10)
ax.legend(fontsize=8); show('data_axis')

> **자료를 아무리 늘려도 초록 선 아래로는 못 내려간다.** 근사 오차는 자료로 살 수 없는 항이다.
> §1.2.5에서 $R^{*}$ 가 자료로 줄지 않는다고 했던 것과 **같은 종류의 벽이지만 다른 벽**이다 —
> 이쪽은 $H$ 를 바꾸면 내려간다.

---
## 5. 계산 축 — 최적화 오차는 음수가 될 수 있다

$k$ 와 $n$ 을 고정하고 경사하강의 걸음 수 $T$ 만 늘린다. $T \to \infty$ 이면 $\hat{f}_T \to f_S$ 이므로
최적화 오차는 0으로 간다. **그런데 도중에 음수를 지난다.**

$w = 0$ 에서 출발한 경사하강은 큰 고유값 방향부터 맞춰 나가고 작은 방향은 나중에 채운다.
작은 방향은 대개 잡음이 지배하므로, **덜 수렴한 해가 정확한 ERM 해보다 참 위험이 낮다.**

In [ ]:
k5, n5 = 12, 30
T5 = 30000 if FAST else 80000
x5, y5 = sample(n5, np.random.default_rng(11))
w5S = erm(x5, y5, k5)
e5S = excess(w5S, k5)
path5, w5T, lr5 = gd_path(x5, y5, k5, T5)
A5 = phi(x5, k5)

print(f"k={k5}, n={n5}, 설계행렬 조건수 {np.linalg.cond(A5):.1f}, 학습률 {lr5:.4f}\n")
print(f"  근사   R(f_H)-R*         = {approx_err(k5):.5f}")
print(f"  정확한 ERM  R(f_S)-R*    = {e5S:.5f}")
print(f"  GD 최소 도달             = {path5.min():.5f}  (T = {int(path5.argmin())})")
print(f"  GD T={T5}                = {path5[-1]:.5f}")
print()
print(f"  최적화 오차 (최소 시점)  = {path5.min()-e5S:+.5f}   <- 음수")
print(f"  최적화 오차 (T={T5})     = {path5[-1]-e5S:+.5f}   <- 0으로 수렴")
print(f"\n  조기 종료가 정확한 ERM보다 {e5S/path5.min():.1f}배 낮은 참 위험을 준다.")

In [ ]:
Ts = np.arange(1, T5+1)
fig, ax = plt.subplots(figsize=(6.2, 4.0))
ax.loglog(Ts, path5, color=CB[5], lw=1.6, label=lab('$R(\\hat{f}_T) - R^{*}$', '$R(\\hat{f}_T) - R^{*}$'))
ax.axhline(e5S, color=CB[4], lw=1.5, ls='--',
           label=lab(f'정확한 ERM $R(f_S)-R^{{*}}$ = {e5S:.3f}', f'exact ERM = {e5S:.3f}'))
ax.axhline(approx_err(k5), color=CB[3], lw=1.5,
           label=lab(f'근사 오차 = {approx_err(k5):.5f}', f'approximation = {approx_err(k5):.5f}'))
ax.plot(path5.argmin()+1, path5.min(), '*', ms=14, color=CB[1],
        label=lab(f'최소 {path5.min():.3f} (T={int(path5.argmin())})', f'minimum at T={int(path5.argmin())}'))
ax.set_xlabel(lab('경사하강 걸음 수 $T$', 'gradient steps $T$'))
ax.set_ylabel(lab('초과 위험', 'excess risk'))
ax.set_title(lab('더 오래 최적화하면 참 위험이 오히려 오른다',
                 'optimizing longer can increase true risk'), fontsize=10)
ax.legend(fontsize=8); show('compute_axis')

> ### ⚠︎ "최적화 오차"라는 이름의 함정
>
> 세 항 중 셋째만 **부호가 정해져 있지 않다.** 그런데 "오차"라 부르면 줄여야 할 것으로 읽힌다.
>
> §1.4.1에서 $H_{\mathrm{reach}}$ 를 도입해 네 항으로 나눴을 때는 세 항이 모두 비음이었다.
> 기준이 $f_S$(정확한 ERM)가 아니라 $f_{\mathrm{re}}$(도달 가능 집합 안의 최선)였기 때문이다.
> **두 분해는 다른 것을 재며, 어느 쪽도 틀리지 않았다.** 무엇을 기준으로 삼았는지를 말하지 않은 진술이 틀린 것이다.
>
> 이 그림이 조기 종료(§10.7)와 암묵적 정칙화(34장)의 최소 사례다.

---
## 6. ⚠︎ 세 항은 따로 조절되지 않는다

3절과 4절을 함께 돌린다. $n$ 마다 총 초과 위험을 최소로 만드는 용량 $k^{\dagger}(n)$ 을 찾는다.

만약 세 항이 독립이라면 최적 용량은 $n$ 과 무관해야 한다 — 근사 오차가 $n$ 을 모르기 때문이다.
**그렇지 않다.**

In [ ]:
NS6 = [20, 40, 80, 160, 320, 640] + ([] if FAST else [1280, 2560])
T6 = 15 if FAST else 40
best_k, best_tot = [], []
for nn in NS6:
    kcap = min(28, max(1, nn//3))
    ks = [kk for kk in range(0, kcap+1)]
    tot = []
    for kk in ks:
        ee = [excess(erm(*sample(nn, np.random.default_rng(300+t)), kk), kk) for t in range(T6)]
        tot.append(float(np.mean(ee)))
    i = int(np.argmin(tot))
    best_k.append(ks[i]); best_tot.append(tot[i])

print("     n   최적 k   그때의 총 초과 위험")
for nn, kk, tt in zip(NS6, best_k, best_tot):
    print(f"{nn:>6}   {kk:>5}   {tt:.6f}")
print("\n-> 자료를 늘리면 최적 용량이 커진다. 근사 오차는 n을 모르는데도 최적점이 움직인다.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].semilogx(NS6, best_k, 'o-', color=CB[4])
axes[0].set_xlabel(lab('표본 수 $n$ (개)', 'sample size $n$'))
axes[0].set_ylabel(lab('총 초과 위험을 최소화하는 $k$', 'capacity minimizing total risk'))
axes[0].set_title(lab('최적 용량은 자료량에 따라 움직인다',
                      'optimal capacity moves with $n$'), fontsize=10)
axes[1].loglog(NS6, best_tot, 'o-', color=CB[0])
axes[1].set_xlabel(lab('표본 수 $n$ (개)', 'sample size $n$'))
axes[1].set_ylabel(lab('그때의 총 초과 위험', 'total excess risk at that $k$'))
axes[1].set_title(lab('용량을 함께 조절했을 때의 도달 가능 위험',
                      'best achievable when capacity is tuned too'), fontsize=10)
show('coupling')

> ### 이 그림이 §1.4.2의 경고를 수치로 확인한다
>
> **근사 오차는 $n$ 을 모릅니다** — $H$ 만의 함수이므로. 그런데 최적 용량은 $n$ 을 따라 움직입니다.
> 세 항이 각각 용량·자료·계산에 대응한다는 표는 **어느 항이 무엇에 반응하는가**를 말할 뿐,
> **셋을 독립적으로 조절할 수 있다**는 뜻이 아닙니다.
>
> 실무적 귀결 하나. "자료를 두 배로 늘렸다"면 **모형 크기도 다시 골라야 합니다.**
> 이전 최적 용량은 더 이상 최적이 아닙니다.

---
## 7. 자기 점검

1. 4절에서 근사 오차가 수평선인 이유를 한 문장으로 설명하라. $k$ 를 바꾸면 그 선은 어디로 가는가?
2. 5절에서 GD가 $T \to \infty$ 에서 $f_S$ 로 수렴하는데도 **최소가 중간에 있는** 이유는? 어떤 방향이 늦게 채워지는가?
3. `SIGMA` 를 0으로 두면 세 항은 각각 어떻게 되는가? 예측한 뒤 확인하라.
4. 참 함수를 $g(x) = |x|$ 에서 다항식(예: $g(x) = x^2$)으로 바꾸면 근사 오차는 어떻게 되는가?

In [ ]:
# 자기 점검 3의 확인 — 잡음을 없애면
print("SIGMA=0 일 때 (k=8, n=100)")
_kk, _nn = 8, 100
_x = np.random.default_rng(5).uniform(-1, 1, _nn)
_y = g(_x)                                  # 잡음 없음
_w = erm(_x, _y, _kk)
print(f"  근사 오차 = {approx_err(_kk):.6f}   (SIGMA와 무관)")
print(f"  추정 오차 = {excess(_w,_kk)-approx_err(_kk):.6f}   (0에 가깝다)")
print("-> 추정 오차는 잡음이 만든다. 근사 오차는 H가 만든다. 서로 다른 원인이다.")

---
## 8. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `SIGMA` | 0절 | 0.3 | 잡음. 추정 오차만 움직이고 근사 오차는 그대로 |
| `g` | 1절 | $\lvert x \rvert$ | 참 함수. 다항식으로 두면 어느 $k$부터 근사 오차가 0 |
| `n3` | 3절 | 200 | 용량 축의 표본 수. 줄이면 U자의 최소점이 왼쪽으로 |
| `k4` | 4절 | 6 | 자료 축의 용량. 근사 오차 수평선의 높이를 정한다 |
| `k5`, `n5` | 5절 | 12, 30 | 계산 축. $n$을 늘리면 조기 종료의 이득이 사라진다 |
| `SAVE_PDF` | 0절 | False | 그림을 벡터 PDF로 저장 |

**권하는 첫 실험** — 5절의 `n5` 를 30에서 200으로 늘리십시오. 최적화 오차의 음수 구간이 **사라집니다.**
정확한 ERM 해가 이미 충분히 좋으면 덜 수렴시켜 얻을 것이 없기 때문이며,
**조기 종료의 이득이 자료량에 의존한다**는 것이 34장의 출발점입니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")